# Punto 1 · Notebook 01 – Exploración de la serie, preparación y modelo recurrente

**Taller 1 – Aprendizaje Profundo** · Daniel Sebastian Velasco Munar

En el notebook anterior (`00`) bajamos los datos de MeteoNet, revisamos la calidad de todas las estaciones de la zona NW y nos quedamos con la **estación 62548002** (cerca de Calais, en la costa norte de Francia, a 3 m sobre el nivel del mar). De ahí salió un CSV horario con 3 años de datos (2016-2018) que es lo que usamos acá.

Este notebook tiene todo el desarrollo del Punto 1, en orden:

1. **Entender la serie** antes de modelar (secciones 1-4): cómo se comporta la temperatura, qué tan fuertes son los ciclos diario y anual, dónde hay huecos, y qué relación tiene con las otras variables. Esto nos sirve para tomar decisiones con argumentos y no a ojo.
2. **Definir la tarea de predicción** (sección 5): cuántas horas hacia atrás mira el modelo y cuántas hacia adelante predice. El taller pide dejar esto explícito.
3. **Preparar los datos** (secciones 6-7): imputar los pocos faltantes, construir las variables de calendario, partir en entrenamiento / validación / test **sin fuga de información**.
4. **Modelar con redes recurrentes** (secciones 8-11): líneas base, comparación SimpleRNN / LSTM / GRU, búsqueda de hiperparámetros con validación, y evaluación final en el 30 % de test con análisis de errores.

Las secciones 1-7 corren en cualquier máquina; de la 8 en adelante conviene **GPU** (en Colab: *Entorno de ejecución > Cambiar tipo de entorno > T4*).


## 0. Preparación del entorno

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

EN_COLAB = "google.colab" in sys.modules
print("Colab:", EN_COLAB)


In [ ]:
# En Colab clonamos el repo para leer el CSV de la estación y dejar las figuras en results/.
REPO_URL = "https://github.com/DANIEL-VELASCO/taller1-deep-learning.git"

if EN_COLAB:
    if not Path("/content/taller1-deep-learning").exists():
        !git clone -q {REPO_URL} /content/taller1-deep-learning
    RAIZ = Path("/content/taller1-deep-learning")
else:
    RAIZ = Path.cwd().resolve().parents[1]     # el notebook vive en code/punto1_rnn_meteonet/

ESTACION = 62548002
RUTA_DATA = RAIZ / "code" / "punto1_rnn_meteonet" / "data"
RUTA_RES = RAIZ / "results" / "punto1"
RUTA_RES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RUTA_DATA / f"estacion_{ESTACION}_horaria.csv", index_col="date", parse_dates=True)
with open(RUTA_DATA / f"estacion_{ESTACION}_metadatos.json", encoding="utf-8") as f:
    meta = json.load(f)

print(f"{len(df):,} filas  |  {df.index.min()}  ->  {df.index.max()}")
meta


## 1. Primer vistazo

Recordatorio de qué es cada columna (ya vienen convertidas a unidades "normales" desde el notebook `00`):

- `t_c`: temperatura (°C). **Es lo que queremos predecir.**
- `td_c`: punto de rocío (°C). Muy ligado a la humedad.
- `hu`: humedad relativa (%).
- `psl_hpa`: presión a nivel del mar (hPa).
- `ff`: velocidad del viento (m/s); `u`, `v`: el mismo viento en componentes este-oeste y norte-sur, para que la dirección no tenga el salto de 359°→0°.
- `precip`: lluvia acumulada en la hora (mm).
- `n_obs`: cuántas observaciones de 6 min había en esa hora (máximo 10). Solo para control de calidad, no entra al modelo.


In [ ]:
display(df.head())
df.describe().T.round(2)


Algo que vale la pena revisar de una: la columna `n_obs`. Si casi siempre es 10, el promedio horario está bien soportado. Si hay muchas horas con 1 o 2 observaciones, esos promedios son ruidosos.

In [ ]:
print(df["n_obs"].value_counts().sort_index())
print(f"\nHoras con las 10 observaciones completas: {(df['n_obs']==10).mean():.1%}")


## 2. Valores faltantes

En el `00` vimos que a esta estación le falta el 0.09% de la temperatura a nivel horario. Es muy poco, pero hay que saber **cómo** falta: no es lo mismo 24 horas sueltas repartidas en 3 años que un día completo sin datos. Lo primero se rellena sin problema interpolando; lo segundo no se puede inventar.

In [ ]:
faltantes = df.isna().mean().sort_values(ascending=False)
print((faltantes * 100).round(3).astype(str) + " %")

# Longitud de los huecos consecutivos en la temperatura
falta_t = df["t_c"].isna()
id_hueco = (~falta_t).cumsum()[falta_t]
huecos = falta_t[falta_t].groupby(id_hueco).agg(["size"])
huecos["inicio"] = falta_t[falta_t].groupby(id_hueco).apply(lambda s: s.index.min())
huecos = huecos.rename(columns={"size": "horas"}).sort_values("horas", ascending=False).reset_index(drop=True)
print(f"\n{len(huecos)} huecos en la temperatura. Los más largos:")
huecos.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 2.2))
ax.eventplot(df.index[df["t_c"].isna()].values, colors="crimson", linelengths=0.8)
ax.set_yticks([]); ax.set_title("Horas sin temperatura a lo largo de los 3 años")
ax.set_xlim(df.index.min(), df.index.max())
fig.savefig(RUTA_RES / "eda_huecos_temperatura.png", bbox_inches="tight"); plt.show()


> **Observaciones (completar después de correr):** ¿cuántos huecos hay y de qué tamaño? ¿están concentrados en una fecha o dispersos? Esto define la estrategia de imputación de la sección 6.

## 3. Cómo se comporta la temperatura

Primero la serie completa. Lo que esperamos ver: el ciclo anual (veranos e inviernos) y, por ser una estación costera, una amplitud relativamente moderada (el mar amortigua los extremos).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
df["t_c"].plot(ax=ax, lw=0.4, color="tab:blue")
df["t_c"].rolling("30D").mean().plot(ax=ax, lw=2, color="tab:red", label="media móvil 30 días")
ax.set_ylabel("°C"); ax.set_xlabel(""); ax.legend(loc="upper left")
ax.set_title(f"Temperatura horaria – estación {ESTACION} (2016-2018)")
fig.savefig(RUTA_RES / "eda_serie_completa.png", bbox_inches="tight"); plt.show()


Ahora un zoom de dos semanas, para ver el ciclo diario "de cerca". Ojo con qué tan regular es: si cada día se ve parecido al anterior, el modelo lo va a tener relativamente fácil; si hay días donde el patrón se rompe (frentes, lluvia), ahí es donde va a fallar.

In [ ]:
zoom = df.loc["2017-07-01":"2017-07-14", "t_c"]
fig, ax = plt.subplots(figsize=(14, 3.5))
zoom.plot(ax=ax, marker=".", ms=3, lw=1)
ax.set_ylabel("°C"); ax.set_xlabel(""); ax.set_title("Dos semanas de julio de 2017")
fig.savefig(RUTA_RES / "eda_zoom_dos_semanas.png", bbox_inches="tight"); plt.show()


### 3.1 Ciclo anual y ciclo diario

Los dos ciclos que dominan una serie de temperatura. El **anual** se ve agrupando por mes; el **diario** agrupando por hora del día. Y como el ciclo diario cambia con la estación (en verano la amplitud día-noche es mayor), lo dibujamos separado por estación del año.

In [ ]:
df["mes"] = df.index.month
df["hora"] = df.index.hour
estaciones_anio = {12: "invierno", 1: "invierno", 2: "invierno", 3: "primavera", 4: "primavera", 5: "primavera",
                   6: "verano", 7: "verano", 8: "verano", 9: "otoño", 10: "otoño", 11: "otoño"}
df["estacion_anio"] = df["mes"].map(estaciones_anio)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

sns.boxplot(data=df, x="mes", y="t_c", ax=axes[0], color="lightsteelblue", fliersize=1)
axes[0].set_title("Ciclo anual: distribución de la temperatura por mes"); axes[0].set_ylabel("°C")

perfil = df.groupby(["estacion_anio", "hora"])["t_c"].mean().unstack(0)[["invierno", "primavera", "verano", "otoño"]]
perfil.plot(ax=axes[1], marker="o", ms=3)
axes[1].set_title("Ciclo diario: temperatura media por hora, según estación del año")
axes[1].set_xlabel("hora del día (UTC)"); axes[1].set_ylabel("°C"); axes[1].set_xticks(range(0, 24, 2))

fig.savefig(RUTA_RES / "eda_ciclos_anual_diario.png", bbox_inches="tight"); plt.show()

amplitud = perfil.max() - perfil.min()
print("Amplitud del ciclo diario (°C):"); print(amplitud.round(2))


> **Observaciones:** ¿cuánto vale la amplitud diaria en verano vs invierno? Comparar con el rango anual (diferencia entre mes más frío y más cálido). Esto da una idea de "cuánto hay que acertar" en cada horizonte: predecir a 24 h exige capturar bien el ciclo diario; predecir a 1 h exige muy poco.

### 3.2 Distribución

Miramos el histograma por dos razones: ver si hay valores absurdos (un sensor dañado se nota como picos raros) y saber si la variable es más o menos simétrica, que es lo que le gusta a una red con `StandardScaler`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
sns.histplot(df["t_c"].dropna(), bins=60, ax=axes[0], color="tab:blue"); axes[0].set_title("Temperatura (°C)")
sns.histplot(df["hu"].dropna(), bins=50, ax=axes[1], color="tab:green"); axes[1].set_title("Humedad (%)")
sns.histplot(df["precip"].dropna().clip(upper=5), bins=50, ax=axes[2], color="tab:gray"); axes[2].set_title("Precipitación (mm/h, recortada a 5)")
fig.savefig(RUTA_RES / "eda_distribuciones.png", bbox_inches="tight"); plt.show()

print(f"Horas con lluvia > 0: {(df['precip'] > 0).mean():.1%}")


## 4. Relación con las otras variables y "memoria" de la serie

Dos preguntas que nos importan para decidir qué entra al modelo y con cuánta historia:

1. ¿Las demás variables aportan información sobre la temperatura? (correlación)
2. ¿Cuánto "se acuerda" la temperatura de sí misma? (autocorrelación). Esto es lo que justifica la **longitud de la ventana de entrada**.

In [ ]:
variables = ["t_c", "td_c", "hu", "psl_hpa", "ff", "u", "v", "precip"]
corr = df[variables].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title("Correlación entre variables")
fig.savefig(RUTA_RES / "eda_correlacion.png", bbox_inches="tight"); plt.show()


La autocorrelación en el rezago *k* mide qué tan parecida es la temperatura de ahora a la de hace *k* horas. En una serie con ciclo diario esperamos picos en 24, 48, 72... horas. Si esos picos se mantienen altos varios días, tiene sentido darle al modelo varios días de historia.

In [ ]:
max_lag = 24 * 8
lags = np.arange(1, max_lag + 1)
acf = np.array([df["t_c"].autocorr(lag=k) for k in lags])

fig, ax = plt.subplots(figsize=(14, 3.6))
ax.plot(lags, acf, lw=1.2)
for d in range(1, 8):
    ax.axvline(24 * d, color="gray", ls=":", lw=0.8)
ax.axvspan(0, 72, color="tab:orange", alpha=0.10, label="ventana de entrada propuesta (72 h)")
ax.set_xlabel("rezago (horas)"); ax.set_ylabel("autocorrelación"); ax.set_xticks(range(0, max_lag + 1, 24))
ax.set_title("Autocorrelación de la temperatura horaria"); ax.legend(loc="lower left")
fig.savefig(RUTA_RES / "eda_autocorrelacion.png", bbox_inches="tight"); plt.show()

print("ACF en múltiplos de 24 h:", {f"{24*d} h": round(float(acf[24*d - 1]), 3) for d in range(1, 8)})


### 4.1 ¿Qué tan difícil es cada horizonte?

Antes de entrenar nada conviene saber cuánto error comete el modelo más tonto posible, la **persistencia**: "la temperatura dentro de *k* horas será igual a la de ahora". Su error crece con *k* y nos da una vara para medir a la RNN. Si la red no le gana a esto, no sirve.

In [ ]:
horizontes = np.arange(1, 49)
mae_persistencia = np.array([(df["t_c"].shift(-k) - df["t_c"]).abs().mean() for k in horizontes])

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(horizontes, mae_persistencia, marker="o", ms=3)
ax.axvline(24, color="tab:red", ls="--", lw=1, label="horizonte propuesto (24 h)")
ax.set_xlabel("horizonte k (horas)"); ax.set_ylabel("MAE de persistencia (°C)")
ax.set_title("Error del modelo de persistencia según el horizonte"); ax.legend()
fig.savefig(RUTA_RES / "eda_dificultad_horizonte.png", bbox_inches="tight"); plt.show()

for k in [1, 3, 6, 12, 24, 48]:
    print(f"k = {k:2d} h  ->  MAE persistencia = {mae_persistencia[k-1]:.2f} °C")


> **Observaciones:** el error de persistencia a 1 h es muy bajo (la temperatura casi no cambia en una hora), sube hasta ~12 h y vuelve a bajar en 24 h por el ciclo diario. Anotar los valores: son la referencia que la RNN tiene que superar.

## 5. Definición de la tarea

Con lo visto arriba, dejamos fijas las decisiones (esto va tal cual al informe):

| Decisión | Valor | Por qué |
|---|---|---|
| Variable objetivo | `t_c` (temperatura en °C) | Lo que pide el taller |
| Frecuencia | horaria | Los datos vienen cada 6 min; a esa escala hay mucho ruido y secuencias larguísimas. Con 1 h se conserva el ciclo diario con 24 puntos |
| **Longitud de entrada** | **72 h** (3 días) | La autocorrelación se mantiene alta en múltiplos de 24 h; con 3 días la red ve tres ciclos diarios y la tendencia reciente. En la sección 10 se prueba también 24, 48 y 168 h |
| **Horizonte** | **24 h**, las 24 horas siguientes de una vez (multi-salida) | Es el problema útil ("mañana") y no es trivial: la persistencia falla bastante. Se compara además con horizonte 1 h como referencia fácil |
| Entradas del modelo | `t_c, td_c, hu, psl_hpa, ff, u, v, precip` + hora y día del año codificados en seno/coseno | Todas las variables observadas más el "reloj", para que la red sepa en qué momento del día y del año está |
| Test | **último 30%** de la serie, en orden cronológico | Lo exige el taller; además es la única forma honesta de evaluar una serie de tiempo |
| Validación | último 15% del 70% restante | Para elegir hiperparámetros y activar los callbacks sin tocar el test |

Esquema de una muestra de entrenamiento:

```
horas:   ... [ h-71  h-70  ...  h-1   h ] [ h+1  h+2  ...  h+24 ] ...
               └──── entrada: 72 h ─────┘  └── salida: 24 h ───┘
               12 variables por hora        solo temperatura
```

Deslizando la ventana una hora cada vez salen unas 26 000 muestras, que se reparten en los tres conjuntos **después** de cortar la serie (nunca una ventana cruza de un conjunto a otro).


## 6. Preparación de los datos

### 6.1 Imputación de faltantes

Regla simple y conservadora: los huecos cortos (hasta 6 horas seguidas) se rellenan con interpolación lineal en el tiempo; los huecos más largos se dejan como `NaN` y más adelante las ventanas que los contienen se descartan. Además marcamos con una bandera qué horas de temperatura fueron imputadas, para poder **excluirlas de las métricas de test**: no tiene sentido evaluar el modelo contra un valor que nos inventamos.

In [ ]:
columnas_modelo = ["t_c", "td_c", "hu", "psl_hpa", "ff", "u", "v", "precip"]
LIMITE_INTERP = 6   # horas

original = df[columnas_modelo].copy()
interpolado = original.interpolate(method="time", limit_area="inside")

# Solo aceptamos la interpolación en huecos de hasta LIMITE_INTERP horas seguidas.
# (el parámetro `limit` de pandas no sirve para esto: rellena las primeras N horas de un hueco largo y deja el resto)
def largo_del_hueco(col):
    falta = col.isna()
    ids = (~falta).cumsum()
    return falta.groupby(ids).transform("sum").where(falta, 0)

for col in columnas_modelo:
    rellenar = original[col].isna() & (largo_del_hueco(original[col]) <= LIMITE_INTERP)
    df[col] = original[col].where(~rellenar, interpolado[col])

df["t_imputada"] = original["t_c"].isna() & df["t_c"].notna()

print(f"Horas de temperatura imputadas: {int(df['t_imputada'].sum())}")
print(f"Horas de temperatura que siguen vacías (huecos > {LIMITE_INTERP} h): {int(df['t_c'].isna().sum())}")
print("\nFaltantes restantes por columna:")
print(df[columnas_modelo].isna().sum())


### 6.2 Variables de calendario

Una red no sabe que la hora 23 y la hora 0 son vecinas. Codificar la hora (y el día del año) como seno y coseno resuelve eso: el "reloj" queda como un punto que da vueltas en un círculo, sin saltos.

In [ ]:
horas_dia = 24
dias_anio = 365.25
t_horas = df.index.hour + df.index.minute / 60
dia = df.index.dayofyear + t_horas / 24

df["hora_sin"] = np.sin(2 * np.pi * t_horas / horas_dia)
df["hora_cos"] = np.cos(2 * np.pi * t_horas / horas_dia)
df["dia_sin"] = np.sin(2 * np.pi * dia / dias_anio)
df["dia_cos"] = np.cos(2 * np.pi * dia / dias_anio)

fig, ax = plt.subplots(figsize=(10, 2.6))
df.loc["2017-01-01":"2017-01-03", ["hora_sin", "hora_cos"]].plot(ax=ax)
ax.set_title("Codificación de la hora del día (3 días de ejemplo)"); ax.set_xlabel("")
plt.show()

FEATURES = columnas_modelo + ["hora_sin", "hora_cos", "dia_sin", "dia_cos"]
print(len(FEATURES), "variables de entrada:", FEATURES)


### 6.3 Partición cronológica: entrenamiento / validación / test

Acá está el punto delicado de la rúbrica ("evitando fuga de información"). Cortamos la serie por **fechas**, en orden: lo primero es entrenamiento, luego validación y el 30% final es test. Nada de barajar. Y guardamos los índices de corte en la configuración.

In [ ]:
n = len(df)
i_test = int(n * 0.70)          # de aquí en adelante es test (30% final)
i_val = int(i_test * 0.85)      # dentro del 70%: 85% entrenamiento, 15% validación

conjuntos = {
    "entrenamiento": df.iloc[:i_val],
    "validacion":    df.iloc[i_val:i_test],
    "test":          df.iloc[i_test:],
}
for nombre, parte in conjuntos.items():
    print(f"{nombre:14s} {len(parte):6,} h  ({len(parte)/n:5.1%})   {parte.index.min()}  ->  {parte.index.max()}")

fig, ax = plt.subplots(figsize=(14, 3.5))
colores = {"entrenamiento": "tab:blue", "validacion": "tab:orange", "test": "tab:green"}
for nombre, parte in conjuntos.items():
    parte["t_c"].plot(ax=ax, lw=0.4, color=colores[nombre], label=nombre)
ax.set_ylabel("°C"); ax.set_xlabel(""); ax.legend(loc="upper left", ncol=3)
ax.set_title("Partición cronológica de la serie")
fig.savefig(RUTA_RES / "eda_particion.png", bbox_inches="tight"); plt.show()


Una consecuencia de partir así que vale la pena decir en el informe: el test va de **febrero a diciembre de 2018**, así que el modelo se evalúa sobre una primavera, un verano y un otoño que nunca vio, pero solo unas tres semanas de invierno. Es una limitación de tener solo 3 años.

### 6.4 Estadísticas para escalar

Las redes entrenan mejor con entradas centradas y con varianza parecida. Usaremos `(x - media) / desviación` por variable, pero **la media y la desviación se calculan solo con el conjunto de entrenamiento**. Si usáramos toda la serie, el modelo "sabría" algo del test aunque sea de forma indirecta. Las guardamos en la configuración.

In [ ]:
stats_train = df.iloc[:i_val][FEATURES].agg(["mean", "std"]).T
stats_train.round(3)


## 7. Guardar la serie preparada y la configuración de la tarea

Dos archivos:

- `estacion_62548002_preparada.csv`: la serie horaria imputada, con las variables de calendario y la bandera de imputación.
- `config_tarea.json`: la definición de la tarea (ventana, horizonte, variables, índices de corte, estadísticas de escalado). Queda como registro de lo decidido y permite reproducir la parte de modelado sin repetir el EDA.

In [ ]:
columnas_guardar = FEATURES + ["t_imputada", "n_obs"]
RUTA_PREP = RUTA_DATA / f"estacion_{ESTACION}_preparada.csv"
df[columnas_guardar].round(4).to_csv(RUTA_PREP)

config = {
    "estacion": ESTACION,
    "objetivo": "t_c",
    "features": FEATURES,
    "largo_entrada_h": 72,
    "horizonte_h": 24,
    "horizonte_referencia_h": 1,
    "frecuencia": "1h",
    "i_val": int(i_val),
    "i_test": int(i_test),
    "fechas": {k: [str(v.index.min()), str(v.index.max())] for k, v in conjuntos.items()},
    "limite_interpolacion_h": LIMITE_INTERP,
    "escalado": {"media": stats_train["mean"].round(6).to_dict(), "std": stats_train["std"].round(6).to_dict()},
    "mae_persistencia_serie_completa": {str(k): round(float(mae_persistencia[k-1]), 4) for k in [1, 3, 6, 12, 24, 48]},
}
with open(RUTA_DATA / "config_tarea.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"guardado {RUTA_PREP.name} ({RUTA_PREP.stat().st_size/1e6:.2f} MB)")
print("guardado config_tarea.json")
print("\nfiguras en results/punto1/:")
for p in sorted(RUTA_RES.glob("eda_*.png")):
    print("  ", p.name)


## 8. Ventanas para la red recurrente

De aquí en adelante entra Keras. Lo primero es convertir la serie en muestras (entrada de `L` horas → salida de `H` horas) como se dibujó en la sección 5. Tres detalles que importan:

- Las ventanas se construyen **dentro de cada conjunto** (entrenamiento, validación, test) por separado, así ninguna muestra mezcla horas de dos conjuntos. Se pierden las primeras `L + H` horas de cada conjunto, que es despreciable.
- Toda ventana que contenga un `NaN` (el hueco de 9 h que no imputamos) se descarta.
- Para el test guardamos además qué horas objetivo fueron imputadas, para dejarlas fuera de las métricas.

Las entradas se escalan con la media y desviación **de entrenamiento** (sección 6.4). La salida también se escala (misma media/desviación de `t_c`); al evaluar se deshace el escalado y todo se reporta en °C.

In [ ]:
import time
import tensorflow as tf
from numpy.lib.stride_tricks import sliding_window_view

print("TensorFlow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus[0].name if gpus else "no hay (va a ser lento)")

SEED = 42
tf.keras.utils.set_random_seed(SEED)

L_BASE, H = config["largo_entrada_h"], config["horizonte_h"]      # 72 y 24
media = np.array([config["escalado"]["media"][f] for f in FEATURES], dtype=np.float32)
desv = np.array([config["escalado"]["std"][f] for f in FEATURES], dtype=np.float32)
IDX_T = FEATURES.index("t_c")
MEDIA_T, DESV_T = float(media[IDX_T]), float(desv[IDX_T])


In [ ]:
def ventanas(parte, L, H):
    """Convierte un tramo de la serie en muestras (X: n×L×F escalado, y: n×H escalado).
    Devuelve también qué objetivos fueron imputados y la hora en que se "emite" cada pronóstico."""
    X = (parte[FEATURES].to_numpy(np.float32) - media) / desv
    y = (parte["t_c"].to_numpy(np.float32) - MEDIA_T) / DESV_T
    imputado = parte["t_imputada"].to_numpy(bool)

    n = len(parte) - L - H + 1
    Xw = sliding_window_view(X, (L, X.shape[1]))[:n, 0]     # ventana i: filas i .. i+L-1
    yw = sliding_window_view(y, H)[L:L + n]                  # objetivo i: filas i+L .. i+L+H-1
    imp_w = sliding_window_view(imputado, H)[L:L + n]
    origen = parte.index[L - 1:L - 1 + n]                    # última hora observada de cada ventana

    ok = ~np.isnan(Xw).any(axis=(1, 2)) & ~np.isnan(yw).any(axis=1)
    return {"X": np.ascontiguousarray(Xw[ok]), "y": np.ascontiguousarray(yw[ok]),
            "imputado": np.ascontiguousarray(imp_w[ok]), "origen": origen[ok]}


def preparar(L, H=H):
    return {nombre: ventanas(parte, L, H) for nombre, parte in conjuntos.items()}


datos_por_L = {L_BASE: preparar(L_BASE)}
for nombre, d in datos_por_L[L_BASE].items():
    print(f"{nombre:14s} X {str(d['X'].shape):18s} y {d['y'].shape}")


## 9. Líneas base

Antes de la primera red, tres modelos "sin aprendizaje" evaluados sobre **las mismas ventanas de test**. Si la RNN no les gana, no está aportando nada:

1. **Persistencia**: las próximas 24 h serán iguales a la última hora observada.
2. **Ayer**: cada hora será igual a la misma hora del día anterior (usa el ciclo diario, que es la estructura más fuerte de la serie).
3. **Climatología**: la temperatura media de esa hora del día y ese mes, calculada solo con entrenamiento.

Como se predicen 24 horas de una vez, el error se promedia sobre los 24 horizontes (no es solo el error "a 24 h").

In [ ]:
def a_grados(y_escalado):
    return y_escalado * DESV_T + MEDIA_T


def metricas(y_real, y_pred, excluir=None):
    """MAE y RMSE en °C (promedio sobre muestras y horizontes) y MAE por horizonte. `excluir` marca objetivos imputados."""
    err = y_pred - y_real
    if excluir is not None:
        err = np.where(excluir, np.nan, err)
    return {"MAE": float(np.nanmean(np.abs(err))),
            "RMSE": float(np.sqrt(np.nanmean(err ** 2))),
            "MAE_h": np.nanmean(np.abs(err), axis=0)}


# Climatología (mes, hora) solo con entrenamiento
train_df = conjuntos["entrenamiento"]
clima = train_df.groupby([train_df.index.month, train_df.index.hour])["t_c"].mean()
clima_tabla = clima.unstack().reindex(index=range(1, 13), columns=range(24)).to_numpy()   # 12 × 24


def pred_climatologia(origen, H):
    objetivos = (origen.values[:, None] + np.arange(1, H + 1) * np.timedelta64(1, "h")).ravel()
    objetivos = pd.DatetimeIndex(objetivos)
    return clima_tabla[objetivos.month.to_numpy() - 1, objetivos.hour.to_numpy()].reshape(len(origen), H)


def lineas_base(d, L):
    y_real = a_grados(d["y"])
    t_in = a_grados(d["X"][:, :, IDX_T])                       # temperatura de la ventana de entrada, en °C
    preds = {
        "persistencia": np.repeat(t_in[:, -1:], H, axis=1),
        "ayer":         t_in[:, L - 24:L],
        "climatologia": pred_climatologia(d["origen"], H),
    }
    return y_real, preds


test_base = datos_por_L[L_BASE]["test"]
y_test_real, preds_base = lineas_base(test_base, L_BASE)
resultados_test = {}
for nombre, p in preds_base.items():
    resultados_test[nombre] = metricas(y_test_real, p, test_base["imputado"])
    print(f"{nombre:14s} MAE = {resultados_test[nombre]['MAE']:.3f} °C   RMSE = {resultados_test[nombre]['RMSE']:.3f} °C")


> **Observaciones:** ¿cuál línea base es la más difícil de vencer? Lo normal es que "ayer" gane a la persistencia en este problema, porque captura el ciclo diario. Ese es el número a superar.

## 10. Red recurrente: diseño y protocolo de experimentación

### 10.1 Arquitectura

Vamos con lo más directo que funciona bien para este tipo de problema: una o dos capas recurrentes (`SimpleRNN`, `LSTM` o `GRU`) que leen las `L` horas de entrada, y una capa `Dense(24)` que produce las 24 horas de salida de una vez. Para regularizar usamos `Dropout` entre capas y, opcionalmente, penalización L2 en los pesos.

Un detalle práctico: no usamos `recurrent_dropout` porque desactiva la implementación cuDNN de LSTM/GRU en GPU y el entrenamiento se vuelve entre 5 y 10 veces más lento; con `Dropout` normal y *early stopping* se controla bien el sobreajuste.

### 10.2 Protocolo

Todas las corridas comparten lo mismo, para que la comparación sea justa:

- Optimizador Adam (lr inicial 1e-3), pérdida MSE sobre la salida escalada, lotes de 128, máximo 60 épocas.
- `EarlyStopping` sobre la pérdida de validación (paciencia 8, se restauran los mejores pesos) y `ReduceLROnPlateau` (baja el lr a la mitad tras 4 épocas sin mejora). Son los callbacks que el taller pide para reaccionar al estancamiento.
- Semilla fija. Aun así, en GPU hay algo de no determinismo; diferencias de centésimas de grado entre corridas no significan nada.
- La selección de hiperparámetros se hace **solo con validación**. El test se toca una vez, al final, con el modelo elegido.

La búsqueda se hace por etapas (una variable a la vez, partiendo de la mejor configuración de la etapa anterior). No es una grilla completa, pero es sistemática y cada decisión queda justificada con un número:

1. Tipo de celda: SimpleRNN vs LSTM vs GRU.
2. Tamaño: unidades × número de capas.
3. Regularización: dropout y L2.
4. Longitud de la ventana de entrada: 24, 48, 72, 168 h.

Cada corrida se anota en `results/punto1/experimentos_rnn.csv`. Si el notebook se vuelve a ejecutar, las corridas ya hechas se leen del archivo y no se repiten.

In [ ]:
EPOCAS, LOTE, PACIENCIA = 60, 128, 8
RUTA_EXP = RUTA_RES / "experimentos_rnn.csv"
CELDAS = {"SimpleRNN": tf.keras.layers.SimpleRNN, "LSTM": tf.keras.layers.LSTM, "GRU": tf.keras.layers.GRU}


def construir_modelo(celda="GRU", unidades=64, capas=1, dropout=0.0, l2=0.0, L=L_BASE, H=H):
    reg = tf.keras.regularizers.l2(l2) if l2 > 0 else None
    modelo = tf.keras.Sequential(name=f"{celda}_{unidades}u_{capas}c")
    modelo.add(tf.keras.Input(shape=(L, len(FEATURES))))
    for i in range(capas):
        modelo.add(CELDAS[celda](unidades, return_sequences=(i < capas - 1), kernel_regularizer=reg))
        if dropout > 0:
            modelo.add(tf.keras.layers.Dropout(dropout))
    modelo.add(tf.keras.layers.Dense(H))
    modelo.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return modelo


def nombre_config(cfg):
    return f"{cfg['celda']}_u{cfg['unidades']}_c{cfg['capas']}_d{cfg['dropout']}_l2{cfg['l2']}_L{cfg['L']}"


def entrenar(cfg, verbose=0):
    """Entrena una configuración y devuelve (modelo, historia, fila de resultados)."""
    if cfg["L"] not in datos_por_L:
        datos_por_L[cfg["L"]] = preparar(cfg["L"])
    d = datos_por_L[cfg["L"]]
    tf.keras.utils.set_random_seed(SEED)
    modelo = construir_modelo(**cfg)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
    ]
    t0 = time.time()
    hist = modelo.fit(d["entrenamiento"]["X"], d["entrenamiento"]["y"],
                      validation_data=(d["validacion"]["X"], d["validacion"]["y"]),
                      epochs=EPOCAS, batch_size=LOTE, callbacks=callbacks, verbose=verbose)
    segundos = time.time() - t0

    fila = dict(cfg)
    fila["id"] = nombre_config(cfg)
    for nombre in ["entrenamiento", "validacion"]:
        pred = a_grados(modelo.predict(d[nombre]["X"], batch_size=512, verbose=0))
        m = metricas(a_grados(d[nombre]["y"]), pred)
        fila[f"mae_{nombre[:5]}"] = round(m["MAE"], 4)
        fila[f"rmse_{nombre[:5]}"] = round(m["RMSE"], 4)
    fila["epocas"] = len(hist.history["loss"])
    fila["mejor_epoca"] = int(np.argmin(hist.history["val_loss"])) + 1
    fila["parametros"] = modelo.count_params()
    fila["segundos"] = round(segundos, 1)
    return modelo, hist, fila


# Registro de experimentos (se reanuda si ya existe el archivo)
tabla = pd.read_csv(RUTA_EXP) if RUTA_EXP.exists() else pd.DataFrame()
modelos, historias = {}, {}


def correr_etapa(etapa, configs):
    global tabla
    for cfg in configs:
        idc = nombre_config(cfg)
        if len(tabla) and idc in set(tabla["id"]):
            print(f"[ya hecho] {idc}")
            continue
        modelo, hist, fila = entrenar(cfg)
        fila["etapa"] = etapa
        modelos[idc], historias[idc] = modelo, hist
        tabla = pd.concat([tabla, pd.DataFrame([fila])], ignore_index=True)
        tabla.to_csv(RUTA_EXP, index=False)
        print(f"{idc:38s} val MAE = {fila['mae_valid']:.3f} °C   train MAE = {fila['mae_entre']:.3f} °C   "
              f"({fila['epocas']} épocas, {fila['segundos']:.0f} s)")
    return tabla[tabla["etapa"] == etapa].sort_values("mae_valid")


def mejor_config(etapa):
    fila = tabla[tabla["etapa"] == etapa].sort_values("mae_valid").iloc[0]
    return {"celda": fila["celda"], "unidades": int(fila["unidades"]), "capas": int(fila["capas"]),
            "dropout": float(fila["dropout"]), "l2": float(fila["l2"]), "L": int(fila["L"])}


COLS_VER = ["id", "mae_entre", "mae_valid", "rmse_valid", "epocas", "mejor_epoca", "parametros", "segundos"]


### 10.3 Etapa 1 – Tipo de celda

Misma configuración (64 unidades, 1 capa, sin regularización, 72 h) cambiando solo la celda recurrente. Con esto respondemos la pregunta básica: ¿la compuerta de LSTM/GRU aporta frente a una RNN simple en secuencias de 72 pasos?

In [ ]:
base = dict(unidades=64, capas=1, dropout=0.0, l2=0.0, L=L_BASE)
etapa1 = correr_etapa("1_celda", [dict(base, celda=c) for c in ["SimpleRNN", "LSTM", "GRU"]])
etapa1[COLS_VER]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for idc in etapa1["id"]:
    if idc in historias:
        h = historias[idc].history
        ax.plot(np.array(h["val_loss"]) ** 0.5 * DESV_T, label=f"{idc.split('_')[0]} (val)")
        ax.plot(np.array(h["loss"]) ** 0.5 * DESV_T, ls="--", alpha=0.5, label=f"{idc.split('_')[0]} (train)")
ax.set_xlabel("época"); ax.set_ylabel("RMSE (°C)"); ax.set_title("Etapa 1: curvas de entrenamiento por tipo de celda")
ax.legend(ncol=3, fontsize=8)
fig.savefig(RUTA_RES / "rnn_etapa1_curvas.png", bbox_inches="tight"); plt.show()


> **Observaciones:** ¿qué celda gana en validación y por cuánto? ¿Se ve sobreajuste (train muy por debajo de val)? ¿Cuántas épocas necesitó cada una antes de que actuara el early stopping?

### 10.4 Etapa 2 – Tamaño de la red

Con la mejor celda, variamos capacidad: unidades ∈ {32, 64, 128} y capas ∈ {1, 2}. Buscamos el punto donde agregar capacidad deja de mejorar validación (o empieza a empeorarla por sobreajuste).

In [ ]:
cfg2 = mejor_config("1_celda")
print("partimos de:", cfg2)
etapa2 = correr_etapa("2_tamano", [dict(cfg2, unidades=u, capas=c) for u in [32, 64, 128] for c in [1, 2]])
etapa2[COLS_VER]


### 10.5 Etapa 3 – Regularización

Con el mejor tamaño, probamos dropout ∈ {0, 0.2, 0.4} y L2 ∈ {0, 1e-4}. Si en la etapa anterior el modelo grande sobreajustaba, aquí debería notarse la ganancia; si no había sobreajuste, la regularización puede incluso empeorar un poco (y eso también es un resultado que vale la pena reportar).

In [ ]:
cfg3 = mejor_config("2_tamano")
print("partimos de:", cfg3)
etapa3 = correr_etapa("3_regularizacion", [dict(cfg3, dropout=d, l2=l) for d in [0.0, 0.2, 0.4] for l in [0.0, 1e-4]])
etapa3[COLS_VER]


### 10.6 Etapa 4 – Longitud de la ventana de entrada

Último hiperparámetro, y uno de los que el taller pide justificar: ¿cuánta historia necesita el modelo? Probamos 24 h (un solo ciclo diario), 48, 72 (nuestra propuesta) y 168 h (una semana).

In [ ]:
cfg4 = mejor_config("3_regularizacion")
print("partimos de:", cfg4)
etapa4 = correr_etapa("4_ventana", [dict(cfg4, L=L) for L in [24, 48, 72, 168]])
etapa4[COLS_VER]


In [ ]:
# Resumen de toda la búsqueda
tabla = tabla.sort_values(["etapa", "mae_valid"])
mejor = tabla.sort_values("mae_valid").iloc[0]
print("Mejor configuración por validación:", mejor["id"], f"-> val MAE {mejor['mae_valid']:.3f} °C")

fig, ax = plt.subplots(figsize=(12, 4))
colores_etapa = {"1_celda": "tab:blue", "2_tamano": "tab:orange", "3_regularizacion": "tab:green", "4_ventana": "tab:red"}
ax.bar(tabla["id"], tabla["mae_valid"], color=tabla["etapa"].map(colores_etapa).tolist())
ax.axhline(resultados_test["ayer"]["MAE"], color="gray", ls="--", lw=1, label='línea base "ayer" (test)')
ax.set_ylabel("MAE validación (°C)"); ax.set_title("Búsqueda de hiperparámetros por etapas")
ax.tick_params(axis="x", rotation=90, labelsize=7); ax.legend()
fig.savefig(RUTA_RES / "rnn_busqueda_hiperparametros.png", bbox_inches="tight"); plt.show()
tabla[["etapa"] + COLS_VER]


> **Observaciones (para la sección de experimentación del informe):** recorrer las cuatro etapas y anotar qué cambió y qué no. Es importante decir también qué **no** funcionó (por ejemplo, si 2 capas o 168 h no aportaron), porque la rúbrica lo pide explícitamente.

## 11. Evaluación final en test

Ahora sí, una sola vez, el modelo elegido contra el 30 % final de la serie (febrero–diciembre 2018), que no se usó para nada hasta este momento. Comparamos contra las tres líneas base sobre las mismas ventanas y excluyendo las horas imputadas.

In [ ]:
cfg_final = {"celda": mejor["celda"], "unidades": int(mejor["unidades"]), "capas": int(mejor["capas"]),
             "dropout": float(mejor["dropout"]), "l2": float(mejor["l2"]), "L": int(mejor["L"])}
id_final = nombre_config(cfg_final)

if id_final not in modelos:          # si el notebook se reanudó, hay que volver a entrenar el elegido
    print("reentrenando el modelo elegido...")
    modelos[id_final], historias[id_final], _ = entrenar(cfg_final)
modelo_final = modelos[id_final]
L_FIN = cfg_final["L"]

d_test = datos_por_L[L_FIN]["test"]
y_test_real, preds_base = lineas_base(d_test, L_FIN)
pred_rnn = a_grados(modelo_final.predict(d_test["X"], batch_size=512, verbose=0))

resultados_test = {n: metricas(y_test_real, p, d_test["imputado"]) for n, p in preds_base.items()}
resultados_test["RNN"] = metricas(y_test_real, pred_rnn, d_test["imputado"])

comparacion = pd.DataFrame({n: {"MAE (°C)": r["MAE"], "RMSE (°C)": r["RMSE"]} for n, r in resultados_test.items()}).T
comparacion["mejora vs ayer"] = (1 - comparacion["MAE (°C)"] / comparacion.loc["ayer", "MAE (°C)"]).map("{:+.1%}".format)
print(f"Modelo final: {id_final}  ({modelo_final.count_params():,} parámetros)")
print(f"Ventanas de test: {len(d_test['y']):,}")
comparacion.round(3)


In [ ]:
modelo_final.save(RUTA_RES / "modelo_rnn_final.keras")
modelo_final.summary()


### 11.1 Error por horizonte

La pregunta natural: ¿el modelo es bueno a 1 h y se degrada a 24 h, o mantiene el error parejo? Y contra las líneas base, ¿en qué horizontes gana más?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
hs = np.arange(1, H + 1)
for nombre, estilo in [("RNN", dict(lw=2.5, color="tab:red")), ("persistencia", dict(ls="--", color="gray")),
                       ("ayer", dict(ls="--", color="tab:blue")), ("climatologia", dict(ls=":", color="tab:green"))]:
    ax.plot(hs, resultados_test[nombre]["MAE_h"], marker="o", ms=3, label=nombre, **estilo)
ax.set_xlabel("horizonte (horas adelante)"); ax.set_ylabel("MAE en test (°C)"); ax.set_xticks(hs[::2])
ax.set_title("Error por horizonte de predicción"); ax.legend()
fig.savefig(RUTA_RES / "rnn_test_error_por_horizonte.png", bbox_inches="tight"); plt.show()

print("MAE de la RNN a 1 h: %.3f °C   6 h: %.3f °C   12 h: %.3f °C   24 h: %.3f °C" % tuple(resultados_test["RNN"]["MAE_h"][[0, 5, 11, 23]]))


### 11.2 ¿Cuándo se equivoca más? Error por mes y por hora de emisión

Agrupamos el error de cada ventana según el mes y según la hora en que se "emitió" el pronóstico. Esto dice, por ejemplo, si el modelo sufre en verano (más amplitud diaria) o si los pronósticos emitidos de madrugada son mejores que los de mediodía.

In [ ]:
err_abs = np.abs(pred_rnn - y_test_real)
err_abs = np.where(d_test["imputado"], np.nan, err_abs)
err_ventana = pd.Series(np.nanmean(err_abs, axis=1), index=d_test["origen"])
err_ayer = pd.Series(np.nanmean(np.where(d_test["imputado"], np.nan, np.abs(preds_base["ayer"] - y_test_real)), axis=1), index=d_test["origen"])

fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
pd.DataFrame({"RNN": err_ventana.groupby(err_ventana.index.month).mean(),
              "ayer": err_ayer.groupby(err_ayer.index.month).mean()}).plot.bar(ax=axes[0], color=["tab:red", "tab:blue"])
axes[0].set_xlabel("mes (2018)"); axes[0].set_ylabel("MAE (°C)"); axes[0].set_title("Error por mes"); axes[0].tick_params(axis="x", rotation=0)

pd.DataFrame({"RNN": err_ventana.groupby(err_ventana.index.hour).mean(),
              "ayer": err_ayer.groupby(err_ayer.index.hour).mean()}).plot(ax=axes[1], marker="o", ms=3, color=["tab:red", "tab:blue"])
axes[1].set_xlabel("hora de emisión del pronóstico (UTC)"); axes[1].set_ylabel("MAE (°C)"); axes[1].set_title("Error según la hora en que se emite")
fig.savefig(RUTA_RES / "rnn_test_error_mes_hora.png", bbox_inches="tight"); plt.show()


### 11.3 Ejemplos concretos

Los números resumen, pero mirar pronósticos individuales es lo que de verdad enseña. Tomamos pronósticos emitidos a medianoche y mostramos el mejor, uno típico (mediana) y el peor día de test: las últimas 48 h observadas, las 24 h reales y lo que dijo la red.

In [ ]:
medianoche = np.where(d_test["origen"].hour == 0)[0]
orden = medianoche[np.argsort(err_ventana.iloc[medianoche].values)]
casos = {"mejor día": orden[0], "día típico (mediana)": orden[len(orden) // 2], "peor día": orden[-1]}

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=False)
for ax, (titulo, i) in zip(axes, casos.items()):
    origen = d_test["origen"][i]
    pasado = pd.Series(a_grados(d_test["X"][i, -48:, IDX_T]), index=origen - pd.to_timedelta(np.arange(47, -1, -1), "h"))
    futuro_idx = origen + pd.to_timedelta(np.arange(1, H + 1), "h")
    ax.plot(pasado.index, pasado.values, color="black", lw=1, label="observado (entrada)")
    ax.plot(futuro_idx, y_test_real[i], color="black", lw=1, ls="--", label="real")
    ax.plot(futuro_idx, pred_rnn[i], color="tab:red", lw=2, label="RNN")
    ax.plot(futuro_idx, preds_base["ayer"][i], color="tab:blue", lw=1, alpha=0.6, label="ayer")
    ax.axvline(origen, color="gray", lw=0.8)
    ax.set_title(f"{titulo}: {origen.date()}  (MAE {err_ventana.iloc[i]:.2f} °C)"); ax.set_xlabel(""); ax.set_ylabel("°C")
axes[0].legend(fontsize=8)
fig.savefig(RUTA_RES / "rnn_test_ejemplos.png", bbox_inches="tight"); plt.show()


### 11.4 Distribución del error y sesgo

Por último: ¿el error es simétrico o el modelo tiende a quedarse corto (por ejemplo, subestimar los picos de la tarde)? Un histograma del error firmado a distintos horizontes lo muestra.

In [ ]:
err_firmado = np.where(d_test["imputado"], np.nan, pred_rnn - y_test_real)
fig, ax = plt.subplots(figsize=(9, 3.8))
for k, color in [(1, "tab:green"), (6, "tab:orange"), (12, "tab:blue"), (24, "tab:red")]:
    sns.kdeplot(err_firmado[:, k - 1][~np.isnan(err_firmado[:, k - 1])], ax=ax, color=color, label=f"{k} h", lw=1.8)
ax.axvline(0, color="black", lw=0.8); ax.set_xlim(-8, 8)
ax.set_xlabel("error (predicho − real), °C"); ax.set_title("Distribución del error por horizonte"); ax.legend(title="horizonte")
fig.savefig(RUTA_RES / "rnn_test_distribucion_error.png", bbox_inches="tight"); plt.show()

sesgo = np.nanmean(err_firmado); p95 = np.nanpercentile(np.abs(err_firmado), 95)
print(f"Sesgo medio: {sesgo:+.3f} °C   |   el 95 % de los errores absolutos está por debajo de {p95:.2f} °C")


### 11.5 Referencia: horizonte de 1 hora

Para tener el punto de comparación "fácil" que mencionamos en la sección 5: la misma arquitectura elegida, pero prediciendo solo la próxima hora. Aquí la persistencia ya es muy buena (0.5-0.6 °C), así que el margen de mejora es pequeño; sirve para mostrar que el problema interesante es el de 24 h.

In [ ]:
datos_h1 = {nombre: ventanas(parte, cfg_final["L"], 1) for nombre, parte in conjuntos.items()}
tf.keras.utils.set_random_seed(SEED)
modelo_h1 = construir_modelo(celda=cfg_final["celda"], unidades=cfg_final["unidades"], capas=cfg_final["capas"],
                             dropout=cfg_final["dropout"], l2=cfg_final["l2"], L=cfg_final["L"], H=1)
modelo_h1.fit(datos_h1["entrenamiento"]["X"], datos_h1["entrenamiento"]["y"],
              validation_data=(datos_h1["validacion"]["X"], datos_h1["validacion"]["y"]),
              epochs=EPOCAS, batch_size=LOTE, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA, restore_best_weights=True),
                         tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5)])

d1 = datos_h1["test"]
y1_real = a_grados(d1["y"])
pred1_rnn = a_grados(modelo_h1.predict(d1["X"], batch_size=512, verbose=0))
pred1_pers = a_grados(d1["X"][:, -1:, IDX_T])
m1_rnn, m1_pers = metricas(y1_real, pred1_rnn, d1["imputado"]), metricas(y1_real, pred1_pers, d1["imputado"])
print(f"Horizonte 1 h en test:  RNN MAE = {m1_rnn['MAE']:.3f} °C   |   persistencia MAE = {m1_pers['MAE']:.3f} °C")


## 12. Resultados guardados y cierre

Dejamos en `results/punto1/` todo lo que necesita el informe: la tabla de experimentos, las métricas de test en JSON, el modelo final y las figuras.

In [ ]:
resumen_final = {
    "estacion": ESTACION,
    "tarea": {"entrada_h": int(L_FIN), "horizonte_h": int(H), "features": FEATURES},
    "modelo_final": cfg_final | {"parametros": int(modelo_final.count_params())},
    "test": {n: {"MAE": round(r["MAE"], 4), "RMSE": round(r["RMSE"], 4),
                 "MAE_por_horizonte": [round(float(x), 4) for x in r["MAE_h"]]} for n, r in resultados_test.items()},
    "test_horizonte_1h": {"RNN_MAE": round(m1_rnn["MAE"], 4), "persistencia_MAE": round(m1_pers["MAE"], 4)},
    "sesgo_medio_C": round(float(sesgo), 4),
    "n_ventanas_test": int(len(d_test["y"])),
    "tensorflow": tf.__version__,
    "gpu": gpus[0].name if gpus else None,
}
with open(RUTA_RES / "metricas_test.json", "w", encoding="utf-8") as f:
    json.dump(resumen_final, f, indent=2, ensure_ascii=False)

print("archivos generados en results/punto1/:")
for p in sorted(RUTA_RES.iterdir()):
    if p.name.startswith(("rnn_", "experimentos", "metricas", "modelo")):
        print("  ", p.name)


## 13. Qué nos llevamos para el informe

Completar con los números que salieron (esto es prácticamente el esqueleto de las secciones de resultados y conclusiones):

**Diseño experimental**
- Estación 62548002 (Calais, costera), 3 años completos, 0.09 % de faltantes en temperatura. Selección justificada con la tabla de 46 candidatas del notebook `00`.
- Partición cronológica 59.5 / 10.5 / 30 %; test = febrero–diciembre 2018. Escalado ajustado solo con entrenamiento; ventanas que no cruzan conjuntos; horas imputadas excluidas de las métricas.
- Tarea: 72 h × 12 variables → 24 h de temperatura (multi-salida). Justificada con la autocorrelación (0.80 a 72 h) y con la curva de dificultad por horizonte.

**Experimentación** (una etapa a la vez, decisión por MAE de validación)
- Celda: ___ ganó a ___ por ___ °C.
- Tamaño: ___ unidades × ___ capas; agregar más ___ (no) ayudó porque ___.
- Regularización: dropout ___ / L2 ___; efecto ___.
- Ventana: ___ h; 168 h ___.
- Lo que no funcionó: ___.

**Resultados en test**
- RNN: MAE ___ °C, RMSE ___ °C. Líneas base: persistencia ___, ayer ___, climatología ___. Mejora relativa vs "ayer": ___ %.
- Error por horizonte: de ___ °C a 1 h hasta ___ °C a 24 h. La RNN gana a las líneas base sobre todo en horizontes ___.
- Meses / horas con más error: ___. Sesgo: ___ (¿subestima los picos?).
- Horizonte 1 h: RNN ___ vs persistencia ___ (poca diferencia, como se esperaba).

**Lecciones aprendidas**
- ___ (por ejemplo: el ciclo diario es tan fuerte que "ayer" es una línea base difícil; la mayor ganancia de la RNN está en ___; la GPU + cuDNN hacen viable la búsqueda en ___ minutos; el early stopping actuó en promedio en la época ___).

**Limitaciones y trabajo futuro**
- El test casi no tiene invierno. Solo una estación. No se probó atención ni modelos autorregresivos. Posible mejora con datos de estaciones vecinas o con el pronóstico numérico (AROME) que también trae MeteoNet.
